
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 15: Arquitecturas de Redes Neuronales y CNN (MNIST)

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

> 💡 **Antes de empezar:** activa GPU en Colab — `Entorno de ejecución > Cambiar tipo de entorno de ejecución > Acelerador de hardware > GPU`. Esto hace que el entrenamiento tome minutos en vez de horas.

## 🎯 Objetivo de la sesión
Entender la arquitectura de una red neuronal desde el perceptrón hasta una CNN, y entrenar un modelo propio para clasificar dígitos escritos a mano (MNIST).

## 🗺️ Tabla de Contenido
1. [Introducción: de ML clásico a Deep Learning](#intro)
2. [El Perceptrón](#perceptron)
3. [Redes Densas (MLP)](#mlp)
4. [El dataset MNIST](#mnist)
5. [Entrenando un MLP sobre MNIST](#mlp-mnist)
6. [Capas Convolucionales y CNN](#cnn)
7. [Entrenando la CNN](#entrenar-cnn)
8. [Evaluación: accuracy, matriz de confusión y errores](#evaluacion)
9. [Ejemplos de aplicación real](#aplicaciones)
10. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Todo lo visto hasta ahora (Regresión, Árboles, Ensambles) son modelos de "Machine Learning clásico". Las **redes neuronales** son otra familia de modelos, inspirada (muy libremente) en cómo funcionan las neuronas biológicas, especialmente potente para datos con estructura compleja como imágenes, audio y texto (esto último lo veremos en la Sesión 16 con LLMs).

<a id="perceptron"></a>
## 2. El Perceptrón

### 🔬 Teoría técnica
El perceptrón (1958) es la neurona artificial más simple: multiplica cada entrada por un peso, suma todo, y aplica una función de activación para decidir la salida.

$$\text{salida} = f\left(\sum_i w_i x_i + b\right)$$

**Limitación clave:** solo puede resolver problemas **linealmente separables** (como una Regresión Logística). No puede, por ejemplo, resolver el problema XOR.

In [ ]:
import numpy as np

def perceptron_simple(entradas, pesos, sesgo):
    suma = np.dot(entradas, pesos) + sesgo
    return 1 if suma > 0 else 0

# Ejemplo dummy: "¿es spam?" según 2 señales (0-1)
# señal 1: contiene "gratis", señal 2: contiene "importante"
pesos = np.array([0.8, -0.6])
sesgo = -0.3

print(perceptron_simple(np.array([1, 0]), pesos, sesgo))  # contiene "gratis" -> probablemente spam
print(perceptron_simple(np.array([0, 1]), pesos, sesgo))  # contiene "importante" -> probablemente no spam

### 🧠 Resumen para dummies
Un perceptrón es una "neurona" que suma entradas ponderadas y decide sí/no. Apilando muchas de estas neuronas en capas, se supera la limitación de solo resolver problemas lineales — eso es exactamente una red neuronal.

<a id="mlp"></a>
## 3. Redes Densas (MLP - Multi-Layer Perceptron)

### 🔬 Teoría técnica
Un MLP apila varias capas de neuronas ("capas ocultas") entre la entrada y la salida. Cada neurona aplica una **función de activación** no lineal:

- **ReLU** (`max(0, x)`): la más usada en capas ocultas, rápida y evita el desvanecimiento de gradiente.
- **Sigmoide**: comprime a [0, 1], útil en la salida de clasificación binaria.
- **Softmax**: convierte varias salidas en probabilidades que suman 1, ideal para clasificación multiclase (como los 10 dígitos de MNIST).

El entrenamiento usa **backpropagation**: se calcula el error de la predicción y se propaga hacia atrás, ajustando los pesos de cada capa para reducir ese error (intuición: "cada neurona recibe una pequeña corrección de responsabilidad por el error final").

<a id="mnist"></a>
## 4. El Dataset MNIST

70,000 imágenes de dígitos escritos a mano (0-9), de 28x28 píxeles en escala de grises. Es el "Hola Mundo" de la visión por computador.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print("Entrenamiento:", X_train.shape, "Prueba:", X_test.shape)

fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Dígito: {y_train[i]}")
    ax.axis("off")
plt.show()

In [ ]:
# Normalizar los píxeles a [0, 1] (equivalente a un Min-Max Scaling, visto en la Sesión 07)
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

<a id="mlp-mnist"></a>
## 5. Entrenando un MLP sobre MNIST

Primero probamos con una red densa simple, "aplanando" la imagen de 28x28 a un vector de 784 valores — esto **ignora** la posición relativa de los píxeles.

In [ ]:
modelo_mlp = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),      # aplana la imagen a un vector de 784
    keras.layers.Dense(128, activation="relu"),        # capa oculta
    keras.layers.Dense(10, activation="softmax"),       # capa de salida: 10 clases (dígitos 0-9)
])

modelo_mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo_mlp.summary()

In [ ]:
historial_mlp = modelo_mlp.fit(
    X_train_norm, y_train, epochs=5, validation_split=0.1, verbose=1,
)

perdida_mlp, accuracy_mlp = modelo_mlp.evaluate(X_test_norm, y_test, verbose=0)
print(f"Accuracy del MLP en prueba: {accuracy_mlp:.4f}")

### 🧠 Resumen para dummies
Un MLP ya funciona bastante bien en MNIST (usualmente >97% accuracy), pero trata cada píxel como un número aislado, sin aprovechar que los píxeles vecinos forman bordes y formas juntos.

<a id="cnn"></a>
## 6. Capas Convolucionales y CNN

### 🔬 Teoría técnica
- **`Conv2D`**: desliza un pequeño filtro (ej. 3x3) sobre la imagen, aprendiendo a detectar patrones locales (bordes, esquinas, texturas). Cada filtro produce un "mapa de características".
- **`MaxPooling2D`**: reduce el tamaño del mapa de características quedándose con el valor máximo de cada región, haciendo el modelo más eficiente y algo más robusto a pequeños desplazamientos.

Al apilar varias capas `Conv2D` + `MaxPooling2D`, la red aprende jerárquicamente: primero bordes simples, luego formas, luego dígitos completos.

In [ ]:
# Las CNN esperan un canal de color explícito (aquí 1, por ser escala de grises)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn = X_test_norm.reshape(-1, 28, 28, 1)

modelo_cnn = keras.Sequential([
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu", input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),   # apaga aleatoriamente 30% de neuronas en entrenamiento, reduce overfitting
    keras.layers.Dense(10, activation="softmax"),
])

modelo_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo_cnn.summary()

### 💪 Fortalezas y debilidades
- **Fortaleza:** aprovecha la estructura espacial de la imagen, generalmente más preciso que un MLP en tareas de visión.
- **Debilidad:** más costoso computacionalmente y con más hiperparámetros que ajustar (número de filtros, tamaño de kernel, etc.).

### 🧠 Resumen para dummies
Una CNN "mira" la imagen en parches pequeños (como si usara una lupa que se desliza), aprendiendo primero bordes, luego formas, y finalmente dígitos completos — capa por capa.

<a id="entrenar-cnn"></a>
## 7. Entrenando la CNN

In [ ]:
historial_cnn = modelo_cnn.fit(
    X_train_cnn, y_train, epochs=5, validation_split=0.1, verbose=1,
)

perdida_cnn, accuracy_cnn = modelo_cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f"Accuracy de la CNN en prueba: {accuracy_cnn:.4f}")
print(f"(Comparar con el MLP: {accuracy_mlp:.4f})")

<a id="evaluacion"></a>
## 8. Evaluación: Accuracy, Matriz de Confusión y Errores

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

predicciones = modelo_cnn.predict(X_test_cnn, verbose=0)
clases_predichas = predicciones.argmax(axis=1)

matriz = confusion_matrix(y_test, clases_predichas)
plt.figure(figsize=(8, 6))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de Confusión — CNN sobre MNIST")
plt.show()

In [ ]:
# Visualizar ejemplos donde el modelo se equivocó
errores = np.where(clases_predichas != y_test)[0]
print(f"Total de errores: {len(errores)} de {len(y_test)}")

fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, idx in enumerate(errores[:5]):
    axes[i].imshow(X_test[idx], cmap="gray")
    axes[i].set_title(f"Real: {y_test[idx]}\nPred: {clases_predichas[idx]}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

### 🧠 Resumen para dummies
La matriz de confusión te dice qué dígitos confunde más el modelo (ej. 4 con 9, o 3 con 8 — son visualmente parecidos incluso para humanos). Revisar los errores manualmente es clave para entender si el modelo falla por un problema real o por dígitos genuinamente ambiguos.

## 🔎 Laboratorio de profundización: pérdida, backpropagation y optimizador

Para clasificación multiclase con softmax se usa entropía cruzada:

$$L=-\frac{1}{n}\sum_i\log p(y_i|x_i)$$

Backpropagation aplica la regla de la cadena para calcular gradientes de la pérdida respecto a cada peso. Un optimizador actualiza:

$$w_{t+1}=w_t-\eta\nabla_w L$$

Adam adapta el tamaño de actualización usando promedios móviles del gradiente; `learning_rate` sigue siendo un hiperparámetro crítico.


In [ ]:
# Paso 1: observar softmax y cross-entropy para una observación
logits = tf.constant([[2.0, 1.0, 0.1]])
clase_real = tf.constant([0])
probabilidades = tf.nn.softmax(logits)
perdida_ce = tf.keras.losses.sparse_categorical_crossentropy(
    clase_real, logits, from_logits=True
)
print("Probabilidades:", probabilidades.numpy())
print("Cross-entropy:", perdida_ce.numpy())


In [ ]:
# Paso 2: una actualización con GradientTape
w_demo = tf.Variable([[0.2], [-0.1]], dtype=tf.float32)
x_demo = tf.constant([[1.0, 2.0]], dtype=tf.float32)
y_demo = tf.constant([[1.0]], dtype=tf.float32)

with tf.GradientTape() as tape:
    pred_demo = tf.matmul(x_demo, w_demo)
    perdida_demo = tf.reduce_mean((y_demo - pred_demo) ** 2)
gradientes = tape.gradient(perdida_demo, [w_demo])
tf.keras.optimizers.SGD(learning_rate=0.1).apply_gradients(zip(gradientes, [w_demo]))
print("Gradiente:", gradientes[0].numpy().ravel())
print("Pesos actualizados:", w_demo.numpy().ravel())


In [ ]:
# Paso 3: separar configuración del modelo y del entrenamiento
optimizador = keras.optimizers.Adam(learning_rate=0.001)
modelo_cnn.compile(
    optimizer=optimizador,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Parámetros entrenables:", modelo_cnn.count_params())
modelo_cnn.summary()


### Hiperparámetros a observar

Arquitectura: filtros, `kernel_size`, `strides`, `padding`, unidades y `dropout`. Entrenamiento: `batch_size`, `epochs`, optimizador y `learning_rate`. `validation_split` no entrena pesos; permite vigilar generalización. Compara curvas `loss`/`val_loss` para detectar subajuste o sobreajuste y considera `EarlyStopping`.


<a id="aplicaciones"></a>
## 9. Ejemplos de Aplicación en el Mundo Real

- Reconocimiento óptico de caracteres (OCR): digitalizar formularios o cheques escritos a mano.
- Clasificación de imágenes médicas (radiografías, resonancias).
- Visión por computador en general: control de calidad industrial, vehículos autónomos, seguridad.

<a id="retos"></a>
## 10. Retos de Práctica

### 🥉 Reto Básico
Entrena un MLP aún más simple (sin la capa oculta de 128 neuronas, solo `Flatten` + `Dense(10, softmax)`) y compara su accuracy contra el MLP y la CNN de este notebook.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Modifica la arquitectura de la CNN agregando una tercera capa `Conv2D` + `MaxPooling2D`, entrénala y compara accuracy y tiempo de entrenamiento contra la CNN original.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Experimenta variando: número de filtros (ej. 16, 32, 64), tasa de `Dropout` (0.2, 0.3, 0.5) y número de épocas (5, 10). Documenta en una tabla cómo cambia el accuracy de prueba y el tiempo de entrenamiento para al menos 3 configuraciones distintas, y visualiza 5 ejemplos donde la mejor configuración se equivoca, proponiendo una hipótesis de por qué.

In [ ]:
# Tu solución al Reto Avanzado aquí
